# PyTorch Training Loop Basics: a plain MLP on Fashion-MNIST

Goal: understand every component of a PyTorch training script by building one piece at a time. The model is a plain feedforward network (MLP): no convolutions.

**The essential components**

| # | Component | PyTorch object |
|---|---|---|
| 1 | Data | `Dataset`, `DataLoader` |
| 2 | Model | `nn.Module` (`nn.Linear`, `nn.ReLU`) |
| 3 | Loss function | `nn.CrossEntropyLoss` |
| 4 | Optimizer | `torch.optim.SGD` |
| 5 | Device | `cpu` / `cuda` |
| 6 | The loop | zero_grad, forward, loss, backward, step |
| 7 | Evaluation | `model.eval()`, `torch.no_grad()` |
| 8 | Saving | `state_dict` |
| 9 | Gradient checks | finite-difference check, grad norms, update ratio, tiny batch test |

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)   # reproducibility
print('PyTorch', torch.__version__)

## 1. Tensors: the basic data structure

Everything in PyTorch is a **tensor**: an n-dimensional array that can live on a GPU and can track gradients.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(x, x.shape, x.dtype)

w = torch.tensor([2.0], requires_grad=True)   # ask PyTorch to track gradients for w
y = (w * 3.0) ** 2                            # y = 9 w^2
y.backward()                                  # autograd computes dy/dw
print('dy/dw =', w.grad.item(), '(expected 18 * w =', 18 * w.item(), ')')

`requires_grad=True` plus `.backward()` is all of **autograd**. Training a network is just this on a bigger function.

## 2. Data: `Dataset` and `DataLoader`

- A `Dataset` returns one `(input, label)` pair at a time.
- A `DataLoader` wraps it to give **mini-batches**, shuffled each epoch.

Each Fashion-MNIST image is 28x28 pixels with a label from 0 to 9. The MLP will flatten it into a 784-number vector, so it never sees "an image", just a list of numbers.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                        # PIL image -> tensor in [0, 1], shape (1, 28, 28)
    transforms.Normalize((0.2860,), (0.3530,)),   # standardise pixel values
])

train_set = datasets.FashionMNIST('data', train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST('data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)    # shuffle only for training
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print(len(train_set), 'train /', len(test_set), 'test images')
print('batches per epoch:', len(train_loader))

In [ ]:
classes = train_set.classes
images, labels = next(iter(train_loader))
print('one batch ->', images.shape, labels.shape)   # (64, 1, 28, 28) and (64,)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for ax, img, lab in zip(axes, images, labels):
    ax.imshow(img.squeeze(), cmap='gray'); ax.set_title(classes[lab], fontsize=8); ax.axis('off')
plt.show()

## 3. Device

Tensors and the model must live on the same device. Use the GPU if there is one.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('using', device)

## 4. The model: subclass `nn.Module`

An `nn.Module` needs two things:
- `__init__`: declare the layers (their weights become trainable **parameters**)
- `forward`: say how an input flows through them

Architecture: `784 -> 128 -> 64 -> 10`. The 10 outputs are raw scores (**logits**), one per class.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden1=128, hidden2=64, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()                   # (B, 1, 28, 28) -> (B, 784)
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)                            # logits: no softmax here (the loss does it)

model = MLP().to(device)
print(model)

In [ ]:
# Parameters: what the optimizer will update
total = 0
for name, p in model.named_parameters():
    print(f'{name:12s} shape={tuple(p.shape)!s:12s} params={p.numel()}')
    total += p.numel()
print('total trainable parameters:', total)

In [ ]:
# A forward pass on one batch (no training yet)
with torch.no_grad():
    logits = model(images.to(device))
print('logits shape:', logits.shape)             # (64, 10)
print('predicted classes:', logits.argmax(dim=1)[:8].tolist())
print('true classes     :', labels[:8].tolist(), '(random weights, so mostly wrong)')

## 5. Loss function and optimizer

- **Loss** (`criterion`): one number saying how wrong the predictions are. For multi-class classification use `nn.CrossEntropyLoss`, which takes logits and integer labels.
- **Optimizer**: updates the parameters using their gradients. Here: plain SGD with momentum.

Sanity check: with 10 balanced classes and random guessing, the starting loss should be about `ln(10) = 2.30`.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

with torch.no_grad():
    print('initial loss:', criterion(model(images.to(device)), labels.to(device)).item(), '(expect about 2.30)')

## 6. One training step, unrolled

The five steps from the slide, run once on a single batch so each effect can be inspected.

In [ ]:
inputs, targets = next(iter(train_loader))
inputs, targets = inputs.to(device), targets.to(device)

w_before = model.fc1.weight[0, :3].detach().clone()

optimizer.zero_grad()                       # 1. clear old gradients
outputs = model(inputs)                     # 2. forward pass
loss = criterion(outputs, targets)          # 3. compute loss
loss.backward()                             # 4. backward pass: fills p.grad for every parameter
print('grad norm of fc1.weight:', model.fc1.weight.grad.norm().item())
optimizer.step()                            # 5. update weights

w_after = model.fc1.weight[0, :3].detach()
print('loss      :', loss.item())
print('weights before:', w_before)
print('weights after :', w_after)

**Why `zero_grad()`?** PyTorch *adds* new gradients to `p.grad` instead of replacing them. Without clearing, gradients from previous batches pile up. Run the cell below to see it.

In [ ]:
p = model.fc3.bias
for i in range(3):
    # deliberately NOT calling zero_grad
    criterion(model(inputs), targets).backward()
    print(f'after {i+1} backward() calls without zero_grad: grad[0] = {p.grad[0].item():.4f}')
optimizer.zero_grad()

## 7. Gradient checking: is learning happening correctly?

Training only works if `loss.backward()` produces gradients that are **correct** and **healthy**. There are two kinds of check:

1. **Correctness:** does the gradient autograd computes match the true slope of the loss? We can measure that slope directly by nudging a weight up and down by a tiny amount (finite differences) and comparing. This is called *gradient checking*. For built-in layers autograd is reliable, so the check mainly protects you when you write a **custom layer, loss or `backward`**, or when something silently cuts the graph (`.detach()`, `.data`, `torch.no_grad()` inside `forward`).
2. **Health:** are the gradients finite, non-zero and a sensible size, and are the weights actually changing by a sensible amount while training?

The seven steps are summarised in a checklist at the end of this section.

### 7.1 The idea: measure the slope by nudging a weight
For one weight `w`, the slope of the loss is approximately

```
(loss(w + eps) - loss(w - eps)) / (2 * eps)
```

Autograd computes the same slope analytically. If the two agree, the backward pass is correct. We do the check in **float64** (float32 is too imprecise for tiny nudges) and on a handful of randomly chosen weights per parameter, since checking all 109,000 would be slow.

In [ ]:
import copy, math
import pandas as pd
from IPython.display import display

def numerical_grad_check(model, inputs, targets, n_checks=5, eps=1e-5, seed=0):
    """Compare autograd's gradient with a finite-difference estimate for a few random weights."""
    model = copy.deepcopy(model).double().cpu().train()      # float64 on CPU: finite differences need precision
    x, y = inputs.double().cpu(), targets.cpu()
    crit = nn.CrossEntropyLoss()

    model.zero_grad()
    crit(model(x), y).backward()                             # the autograd gradient

    g = torch.Generator().manual_seed(seed)
    rows = []
    for name, p in model.named_parameters():
        for _ in range(n_checks):
            idx = tuple(torch.randint(0, s, (1,), generator=g).item() for s in p.shape)   # random weight
            analytic = 0.0 if p.grad is None else p.grad[idx].item()
            orig = p.data[idx].item()
            with torch.no_grad():
                p.data[idx] = orig + eps; loss_plus  = crit(model(x), y).item()
                p.data[idx] = orig - eps; loss_minus = crit(model(x), y).item()
                p.data[idx] = orig                                                       # restore
            numeric = (loss_plus - loss_minus) / (2 * eps)
            rel_err = abs(analytic - numeric) / max(abs(analytic) + abs(numeric), 1e-8)
            rows.append(dict(param=name, index=idx, autograd=analytic, numerical=numeric, rel_error=rel_err))
    return pd.DataFrame(rows)

### 7.2 Check our model

`rel_error` is the relative difference between the two gradients. In float64 with a correct backward pass it is typically far below `1e-5` (often around `1e-8` or smaller). Weights whose gradient is exactly 0 for both (for example behind a ReLU that is switched off) count as a match.

In [ ]:
inputs16, targets16 = next(iter(train_loader))
inputs16, targets16 = inputs16[:16], targets16[:16]

check = numerical_grad_check(MLP(), inputs16, targets16)
display(check.head(10))
print('max relative error over', len(check), 'checked weights:', f'{check.rel_error.max():.2e}')
print('GRADIENT CHECK:', 'PASS' if check.rel_error.max() < 1e-5 else 'FAIL')

### 7.3 Does the check catch a real bug?

A very common silent bug: something in `forward` cuts the autograd graph, so the layers before that point **never learn**, yet no error is raised and the loss may still go down a little (the later layers still train). Here we `.detach()` the output of the first hidden layer.

In [ ]:
class DetachedMLP(MLP):
    def forward(self, x):
        x = self.flatten(x)
        x = self.act(self.fc1(x)).detach()          # BUG: cuts the graph, so fc1 gets no gradient
        x = self.act(self.fc2(x))
        return self.fc3(x)

bad = numerical_grad_check(DetachedMLP(), inputs16, targets16)
print(bad.groupby('param').rel_error.max().to_frame('max rel_error'))
print('GRADIENT CHECK:', 'PASS' if bad.rel_error.max() < 1e-5 else 'FAIL  <- caught: fc1 shows autograd = 0 but a real nonzero slope')

`fc1.weight` and `fc1.bias` fail (relative error near 1): autograd says the slope is 0, but nudging the weights clearly changes the loss. The numerical check sees what autograd was told to ignore.

### 7.4 Gradient checking for custom autograd functions

If you write your own operation with a hand-written `backward`, PyTorch ships a built-in checker, `torch.autograd.gradcheck`. Below, `Square` has a correct backward (`d/dx x^2 = 2x`). `BuggySquare` forgets the factor 2.

In [ ]:
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 2
    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out * 2 * x                      # correct

class BuggySquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 2
    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out * x                          # BUG: forgot the factor 2

x = torch.randn(5, dtype=torch.double, requires_grad=True)       # gradcheck needs float64
print('Square      gradcheck passes:', torch.autograd.gradcheck(Square.apply, (x,)))
print('BuggySquare gradcheck passes:', torch.autograd.gradcheck(BuggySquare.apply, (x,), raise_exception=False))

### 7.5 Gradient health after one backward pass

Even with correct gradients, training can fail because of *what* the gradients are. After one `backward()`, look at every parameter:

- `has_grad`: `False` means the parameter is not connected to the loss (frozen, or a detached branch).
- `finite`: `False` means NaN or inf, and training is about to blow up.
- `grad_norm`: should be neither 0 nor enormous.
- `frac_zero`: a very high fraction of exactly-zero gradients suggests many dead ReLU units.

In [ ]:
def gradient_report(model):
    rows = []
    for name, p in model.named_parameters():
        g = p.grad
        rows.append(dict(
            param=name,
            has_grad=g is not None,
            finite=bool(torch.isfinite(g).all()) if g is not None else None,
            grad_norm=g.norm().item() if g is not None else None,
            frac_zero=(g == 0).float().mean().item() if g is not None else None,
            weight_norm=p.norm().item()))
    return pd.DataFrame(rows).set_index('param')

fresh = MLP().to(device)
x_b, y_b = next(iter(train_loader))
fresh.zero_grad()
criterion(fresh(x_b.to(device)), y_b.to(device)).backward()
display(gradient_report(fresh).round(4))

### 7.6 Monitor gradients and updates while training

During a real run, log three things every step:

- **loss:** should trend down.
- **global gradient norm:** should be finite and fairly stable. Sudden spikes mean instability (a lower learning rate or gradient clipping may help). Vanishing to ~0 means learning has stalled.
- **update ratio per layer**, `||change in weights|| / ||weights||`: a common rule of thumb is that it should sit around `1e-3` (roughly `1e-4` to `1e-2`). Far below means that layer is barely learning (learning rate too low). Far above means unstable updates.

The loop also stops immediately if the loss or gradient becomes NaN or inf. Note that `clip_grad_norm_(..., max_norm=inf)` is a handy trick: it **measures** the global norm without clipping anything.

In [ ]:
import matplotlib.pyplot as plt

def monitored_training(steps=300, lr=0.01, seed=0):
    torch.manual_seed(seed)
    m = MLP().to(device)
    opt = optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    weight_names = [n for n, _ in m.named_parameters() if n.endswith('weight')]
    log = dict(loss=[], grad_norm=[], **{n: [] for n in weight_names})
    it = iter(train_loader)
    for step in range(steps):
        try: xb, yb = next(it)
        except StopIteration:
            it = iter(train_loader); xb, yb = next(it)
        xb, yb = xb.to(device), yb.to(device)
        before = {n: p.detach().clone() for n, p in m.named_parameters() if n in weight_names}

        opt.zero_grad()                                           # 1. clear gradients
        loss = criterion(m(xb), yb)                               # 2-3. forward + loss
        loss.backward()                                           # 4. backward
        gnorm = torch.nn.utils.clip_grad_norm_(m.parameters(), float('inf')).item()   # measure, do not clip
        assert math.isfinite(loss.item()) and math.isfinite(gnorm), f'non-finite loss or gradient at step {step}'
        opt.step()                                                # 5. update

        log['loss'].append(loss.item()); log['grad_norm'].append(gnorm)
        for n, p in m.named_parameters():
            if n in before:
                log[n].append(((p.detach() - before[n]).norm() / before[n].norm()).item())
    return log, weight_names

log, weight_names = monitored_training()

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].plot(log['loss']); ax[0].set_title('loss'); ax[0].set_xlabel('step')
ax[1].plot(log['grad_norm'], color='C1'); ax[1].set_title('global gradient norm'); ax[1].set_xlabel('step')
for n in weight_names:
    ax[2].plot(log[n], label=n, lw=1)
ax[2].axhline(1e-3, color='k', ls='--', lw=1, alpha=.5)
ax[2].set_yscale('log'); ax[2].set_title('update ratio  ||dW|| / ||W||  (dashed: 1e-3)'); ax[2].set_xlabel('step'); ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"loss: {log['loss'][0]:.3f} -> {sum(log['loss'][-20:]) / 20:.3f} (mean of last 20 steps)")

### 7.7 The tiny batch test

The final and most practical check: train on just **32 fixed images** for a few hundred steps. A healthy pipeline can memorize them, so the loss must go to about 0 and accuracy to 100%. If it can't, a bug is hiding somewhere in the data, model, loss or loop, which is far cheaper to find now than after a full training run.

In [ ]:
def overfit_tiny_batch(n=32, steps=300, lr=0.01, seed=0):
    xs = torch.stack([train_set[i][0] for i in range(n)]).to(device)      # ONE fixed batch
    ys = torch.tensor([train_set[i][1] for i in range(n)]).to(device)
    torch.manual_seed(seed)
    m = MLP().to(device)
    opt = optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        out = m(xs)
        loss = criterion(out, ys)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    acc = (m(xs).argmax(1) == ys).float().mean().item()
    return losses, acc

losses, acc = overfit_tiny_batch()
print(f'loss: {losses[0]:.3f} -> {losses[-1]:.4f} | accuracy on the {32} images: {acc:.0%}')
print('TINY BATCH TEST:', 'PASS' if (losses[-1] < 0.1 and acc == 1.0) else 'FAIL, debug the pipeline before full training')

### 7.8 Checklist: the steps to use on any new model

| # | Step | What to expect | If it fails |
|---|---|---|---|
| 1 | **Initial loss** on one batch | about `ln(num_classes)` (2.30 for 10 classes) | wrong loss/labels, bad init, softmax applied twice |
| 2 | **Numerical gradient check** (float64, small batch, few random weights per parameter) | relative error `< 1e-5` | wrong custom `backward`, graph cut by `.detach()` / `.data` / `no_grad` |
| 3 | **Gradient health after one `backward()`** | every parameter has a gradient; all finite; norms not 0 and not huge | frozen or disconnected layers, NaN/inf, dead ReLUs |
| 4 | **Weights actually change** after `optimizer.step()` | update ratio about `1e-3` | missing `step()`, learning rate too low or too high |
| 5 | **Tiny batch overfit** (32 images) | loss to about 0, accuracy 100% | a bug in the data, model, loss or loop |
| 6 | **Monitor during training** (loss, global grad norm, update ratios, NaN guard) | loss trending down; stable gradient norm | spikes: lower lr or clip gradients; flat: learning stalled |
| 7 | **Then train on the full data** and compare train vs test curves | steady learning, sensible gap | see the sections on learning rate, epochs and network size |

Steps 2 to 5 take seconds. Do them once when you build or change a model, not on every run.

## 8. Evaluation function

Two differences from training:
- `model.eval()` turns off training-only behaviour (dropout, batch-norm updates). Our MLP has neither, but always switch anyway; it is the habit that matters.
- `torch.no_grad()` skips building the autograd graph, which saves memory and time.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item() * inputs.size(0)   # undo the batch mean
            correct += (outputs.argmax(dim=1) == targets).sum().item()
            n += inputs.size(0)
    return total_loss / n, correct / n

print('before training: loss=%.3f  acc=%.3f' % evaluate(model, test_loader))

## 9. The full training loop

Put steps 1-5 inside a loop over mini-batches, and that inside a loop over epochs. One **epoch** is one full pass over the training data.

In [ ]:
# fresh model so the run starts from scratch
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

num_epochs = 10
history = {'train_loss': [], 'test_loss': [], 'test_acc': []}

for epoch in range(num_epochs):
    model.train()                                   # training mode
    running_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)   # move data to the device

        optimizer.zero_grad()                       # 1. zero out accumulated gradients
        outputs = model(inputs)                     # 2. forward pass
        loss = criterion(outputs, targets)          # 3. calculate loss
        loss.backward()                             # 4. backward pass
        optimizer.step()                            # 5. update weights

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_set)
    test_loss, test_acc = evaluate(model, test_loader)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    print(f'epoch {epoch+1:2d}/{num_epochs}  train_loss={train_loss:.4f}  test_loss={test_loss:.4f}  test_acc={test_acc:.4f}')

`.item()` converts a one-element tensor to a Python number. Without it, `running_loss` would keep the autograd graph alive and leak memory.

## 10. Learning curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ep = range(1, num_epochs + 1)
ax[0].plot(ep, history['train_loss'], marker='o', label='train')
ax[0].plot(ep, history['test_loss'],  marker='o', label='test')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, history['test_acc'], marker='o', color='green')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('test accuracy'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 11. Looking at predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    probs = torch.softmax(model(images.to(device)), dim=1).cpu()   # logits -> probabilities
pred = probs.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
for ax, img, y, p, pr in zip(axes.flat, images, labels, pred, probs):
    ax.imshow(img.squeeze(), cmap='gray'); ax.axis('off')
    ax.set_title(f'{classes[p]}\n{pr[p]:.0%}', fontsize=8, color='green' if p == y else 'red')
plt.suptitle('prediction and confidence (green = correct, red = wrong)'); plt.show()

## 12. Saving and loading

Save the **`state_dict`** (a dict of parameter tensors), not the whole model object. To load, build the same architecture first, then load the weights.

In [ ]:
torch.save(model.state_dict(), 'fashion_mlp.pt')

restored = MLP().to(device)
restored.load_state_dict(torch.load('fashion_mlp.pt', map_location=device))
print('restored model: loss=%.4f  acc=%.4f' % evaluate(restored, test_loader))

## Recap

```
for epoch in range(num_epochs):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()                  # 1. clear gradients
        outputs = model(inputs)                # 2. forward
        loss = criterion(outputs, targets)     # 3. loss
        loss.backward()                        # 4. backward
        optimizer.step()                       # 5. update
    evaluate(...)                              # model.eval() + torch.no_grad()
```

**Common mistakes to discuss**
- Forgetting `optimizer.zero_grad()` (gradients accumulate)
- Model and data on different devices
- Forgetting `model.eval()` / `torch.no_grad()` at evaluation time
- Applying `softmax` before `CrossEntropyLoss` (it already includes it)
- Accumulating `loss` instead of `loss.item()`
- Never verifying gradients: a wrong custom `backward`, or an accidental `.detach()` / `.data` / `torch.no_grad()` in `forward`, fails silently

## Exercises
1. Change `lr` to 0.001 and 0.5. What happens to the curves? Why?
2. Change `batch_size` to 8 and to 2048. How do the time per epoch and the curves change?
3. Replace SGD with `optim.Adam(model.parameters(), lr=1e-3)`.
4. Comment out `optimizer.zero_grad()` in the loop and observe training.
5. Add `nn.Dropout(0.3)` after each hidden activation. Why does `model.eval()` now matter?
6. Add a `torch.optim.lr_scheduler.StepLR` and call `scheduler.step()` once per epoch.
7. In section 7, move the `.detach()` from `fc1` to `fc2` in `DetachedMLP`. Which layers now fail the gradient check?
8. Run `monitored_training(lr=0.5)` and `monitored_training(lr=0.0001)`. How do the update ratios and gradient norm change? When does the NaN guard trigger?
9. Write your own `Cube` autograd function (forward `x**3`, backward `3x^2`), and use `gradcheck` to verify it.